In [ ]:
from theia.data_loading import load_bakom_ukw_transmitters


txs = load_bakom_ukw_transmitters()
txs = [tx for tx in txs if tx.power >= 2000]

In [ ]:
import datetime
import itertools

import numpy as np

from theia.coordinates import POSITIONS_OF_INTEREST
from theia.detection.pcl import PclDetector
from theia.terrain import AbstractTerrainModel, SrtmTerrainModel
from theia.test_data import build_pcl_receiver, build_single_target_from_Bodensee
from theia.types import Point, Target, Transmitter

terrain = SrtmTerrainModel()
detector = PclDetector()

origin_lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
origin_lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]
origin_alt = terrain.elevationAt(origin_lat, origin_lon)
p0 = Point(lat=origin_lat, lon=origin_lon, alt=origin_alt)

trajectory = build_single_target_from_Bodensee()
targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))

roi_lat_min = 46.3631
roi_lon_min = 8.1625
roi_lat_max = 48.0451
roi_lon_max = 9.2996

txs_roi = [
    tx
    for tx in txs
    if roi_lat_min <= tx.lat <= roi_lat_max and roi_lon_min <= tx.lon <= roi_lon_max
]


def f(
    lat: float,
    lon: float,
    detector: PclDetector = PclDetector(),
    txs: list[Transmitter] = txs_roi,
    targets: list[Target] = targets,
    terrain: AbstractTerrainModel = terrain,
) -> float:
    rx = build_pcl_receiver(
        0,
        Point(lat=lat, lon=lon, alt=terrain.elevationAt(lat, lon)),
    )
    value_sum = 0.0
    for target in targets:
        snrs = []
        for tx in txs:
            try:
                snrs.append(detector.calculate_raw_measurement(rx, tx, target)[0])
            except:
                pass
        snrs = sorted(snrs)
        value_sum += sum(snrs[-3:])
    return value_sum

In [ ]:
# %%timeit
# f(p0.lat, p0.lon)

In [ ]:
from bayes_opt import BayesianOptimization

pbounds = {
    "lat": (roi_lat_min, roi_lat_max),
    "lon": (roi_lon_min, roi_lon_max),
}

n_init = 30

optimizer = BayesianOptimization(
    f=f,
    pbounds=pbounds,
    verbose=0,  # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=1,
)
optimizer.maximize(
    init_points=n_init,
    n_iter=60,
)

In [ ]:
x_obs = optimizer.space.params
y_obs = optimizer.space.target

In [ ]:
GRID_N = 50
lats = np.linspace(*pbounds["lat"], GRID_N)
lons = np.linspace(*pbounds["lon"], GRID_N)
XX, YY = np.meshgrid(lats, lons)
XY_flat = np.column_stack([XX.ravel(), YY.ravel()])

ground_truth = [f(lat, lon) for lat, lon in itertools.product(lats, lons)]
gt = np.array(ground_truth).reshape((GRID_N, GRID_N))

In [ ]:
gp = optimizer._gp
MU, SIGMA = gp.predict(XY_flat, return_std=True)
MU    = MU.reshape(XX.shape)
SIGMA = SIGMA.reshape(XX.shape)

In [ ]:
p_max = (optimizer.max["params"]["lat"], optimizer.max["params"]["lon"])

In [ ]:
from matplotlib import pyplot as plt


fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 4.5))
# img = ax.contourf(XX, YY, MU, levels=60)
axes[0].contourf(XX, YY, MU, cmap="viridis", levels=12)
axes[1].contourf(XX, YY, gt, levels=12, cmap="viridis")

axes[0].plot([p_max[0]], [p_max[1]], "*")
axes[1].plot([p_max[0]], [p_max[1]], "*")

# TRUE_Z  = target(XX, YY)

In [ ]:
from matplotlib import pyplot as plt


fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(optimizer.space.target)

ax.axvline(n_init, color="orange")
ax.axhline(15)

In [ ]:
import shapely

import theia
from theia.detection.pcl import pcl_track_init_update_masks
from theia.grids import LatLonHeightGrid
from theia.simulation.server import GeoJSONFeature, GeoJSONMultiPolygon
from theia.types import PclSensor
from theia.util import mask_to_polygon


def calculate_pcl_coverage(
        sensors: list[PclSensor],
        grid: LatLonHeightGrid,
        rcs: float,
        snr_threshold: float = theia.config.SNR_THRESHOLD_PCL,
        doppler_threshold: float = theia.config.DOPPLER_SHIFT_THRESHOLD_PCL,
        delay_threshold: float = theia.config.DELAY_THRESHOLD_PCL,
    ) -> tuple[GeoJSONFeature, GeoJSONFeature]:
        """
        Calculate PCL coverage.

        Parameters
        ----------
        sensors: list[PclSensor]
            Sensors
        grid: LatLonHeightGrid
            Calculation grid
        rcs: float
            Radar cross section for which to calculate the coverage
        snr_threshold: float, default theia.config.SNR_THRESHOLD_PCL
            Minimum detectable threshold [dB]
        doppler_threshold: float, default theia.config.DOPPLER_SHIFT_THRESHOLD_PCL
            Minimum detectable Doppler shift [Hz]
        delay_threshold: float, default theia.config.DELAY_THRESHOLD_PCL
            Delay threshold for PCL [us].
            This is used to judge whether a given transmitter - target - receiver geometry
            is in the bistatic or the forward scattering regime.

        Returns
        -------
        track_init_coverage: GeoJSONFeature
            Region in which a track init can happen only using PCL
        track_update_coverage: GeoJSONFeature
            Region in which a track update can happen only using PCL
        """
        if len(sensors) == 0:
            return GeoJSONFeature.from_shapely(
                shapely.Polygon()
            ), GeoJSONFeature.from_shapely(shapely.Polygon())
        assert grid.altitude_values.shape[0] == 1

        detector = PclDetector(
            snr_threshold=snr_threshold,
            doppler_threshold=doppler_threshold,
            delay_threshold=delay_threshold,
        )

        track_init_mask, track_update_mask = pcl_track_init_update_masks(
            detector,
            sensors,
            grid,
            rcs,
        )

        polygons_init = mask_to_polygon(
            track_init_mask[:, :, 0],
            grid.latitude_values[0],
            grid.latitude_values[1] - grid.latitude_values[0],
            grid.longitude_values[0],
            grid.longitude_values[1] - grid.longitude_values[0],
        )

        polygons_update = mask_to_polygon(
            track_update_mask[:, :, 0],
            grid.latitude_values[0],
            grid.latitude_values[1] - grid.latitude_values[0],
            grid.longitude_values[0],
            grid.longitude_values[1] - grid.longitude_values[0],
        )
        return (
            GeoJSONFeature(
                geometry=GeoJSONMultiPolygon.from_shapely(
                    shapely.MultiPolygon(polygons_init)
                )
            ),
            GeoJSONFeature(
                geometry=GeoJSONMultiPolygon.from_shapely(
                    shapely.MultiPolygon(polygons_update)
                )
            ),
        )

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
folium.Marker(p_max, icon=folium.Icon(color="darkgreen")).add_to(map)
for tx in txs:
    folium.Marker((tx.lat, tx.lon), tooltip=f"Tx ID = {tx.id}").add_to(map)
for target in targets:
    folium.Marker((target.lat, target.lon), icon=folium.Icon(color="red")).add_to(map)
map

In [ ]:
47.0658, 8.5250

In [ ]:
optimizer.logger

In [ ]:
import numpy as np
import pydantic

from theia.coordinates import POSITIONS_OF_INTEREST, CoordinateTransformations
from theia.terrain import FlatEarthTerrainModel, PlateauHill
from theia.types import Point


class FlatEarthCoordinateSystem(pydantic.BaseModel):
    origin: Point
    terrain_model: FlatEarthTerrainModel

    def model_post_init(self, context):
        self._origin_ecef = np.array(
            CoordinateTransformations.geodetic_to_cartesian(
                self.origin.lat,
                self.origin.lon,
                self.origin.alt,
            )
        )

    def flat_earth_to_ecef(
        self,
        x: float,
        y: float,
        z: float,
    ) -> tuple[float, float, float]:
        R = np.array(self.terrain_model.plane_directions)
        print(np.max(np.abs(R @ R.T - np.eye(3))))
        print(np.max(np.abs(R.T @ R - np.eye(3))))
        p_ecef = R.T @ np.array((x, y, z)) + self._origin_ecef
        return tuple(p_ecef)

    def flat_earth_to_geodetic(
        self,
        x: float,
        y: float,
        z: float,
    ) -> Point:
        p_ecef = self.flat_earth_to_ecef(x, y, z)
        p = CoordinateTransformations.cartesian_to_geodetic(*p_ecef)
        return Point(
            lat=p[0],
            lon=p[1],
            alt=p[2],
        )


def build_flat_coord_wall(
    bottom_left: tuple[float, float],
    top_right: tuple[float, float],
    height: float,
    coordinate_system: FlatEarthCoordinateSystem,
) -> PlateauHill:
    p1 = coordinate_system.flat_earth_to_geodetic(bottom_left[0], bottom_left[1], 0)
    p2 = coordinate_system.flat_earth_to_geodetic(top_right[0], top_right[1], 0)

    lat_min = min(p1.lat, p2.lat)
    lat_max = max(p1.lat, p2.lat)
    lon_min = min(p1.lon, p2.lon)
    lon_max = max(p1.lon, p2.lon)

    return PlateauHill(
        lat_min=lat_min,
        lat_max=lat_max,
        lon_min=lon_min,
        lon_max=lon_max,
        height=height,
    )


In [ ]:
from theia.terrain import FlatEartWithHillsTerrainModel

flat_terrain = FlatEarthTerrainModel()

origin_lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
origin_lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]
origin_alt = flat_terrain.elevationAt(origin_lat, origin_lon)
origin = Point(lat=origin_lat, lon=origin_lon, alt=origin_alt)

extent_lat = 8.0
extent_lon = 8.0

system = FlatEarthCoordinateSystem(origin=origin, terrain_model=flat_terrain)

plateau1 = build_flat_coord_wall(
    (-200_000, -75_000),
    (
        -198_000,
        425_000,
    ),
    1000,
    system,
)
plateau2 = build_flat_coord_wall(
    (-200_000, -75_000),
    (500_000, -73_000),
    1000,
    system,
)

terrain = FlatEartWithHillsTerrainModel(plateaus=[plateau1])

# Sanity check.
assert np.allclose(
    system.flat_earth_to_geodetic(0, 0, 0).as_tuple(),
    origin.as_tuple(),
)

In [ ]:
p_monostatic1_flat = (0, 0, 0)
p_monostatic2_flat = (200_000, 0, 0)
p_monostatic3_flat = (0, 100_000, 0)

p_monostatic1 = system.flat_earth_to_geodetic(*p_monostatic1_flat)
p_monostatic2 = system.flat_earth_to_geodetic(*p_monostatic2_flat)
p_monostatic3 = system.flat_earth_to_geodetic(*p_monostatic3_flat)

In [ ]:
p_monostatic1 = system.flat_earth_to_ecef(*p_monostatic1_flat)
p_monostatic2 = system.flat_earth_to_ecef(*p_monostatic2_flat)
p_monostatic3 = system.flat_earth_to_ecef(*p_monostatic3_flat)

points = np.array((p_monostatic1, p_monostatic2, p_monostatic3))

d1 = points[1, :] - points[0, :]
d2 = points[2, :] - points[0, :]

np.sqrt(abs(np.dot(d1, d2)))

# import plotly.express as px
# px.scatter_3d(x=points[:, 0], y=points[:, 1], z=points[:, 2])

In [ ]:
from theia.export_paraview import ParaviewExporter, PointOfInterest


rotation_center = np.mean(np.array(terrain.corners_ecef), axis=0)
rotation_center = Point(
    lat=rotation_center[0],
    lon=rotation_center[1],
    alt=rotation_center[2],
)

exporter = ParaviewExporter(
    "output/terrain_with_hills",
    lat_min=origin.lat - extent_lat / 2.0,
    lat_max=origin.lat + extent_lat / 2.0,
    lat_res=0.01,
    lon_min=origin.lon - extent_lon / 2.0,
    lon_max=origin.lon + extent_lon / 2.0,
    lon_res=0.01,
    terrain_model=terrain,
    elevation_factor=10.0,
    fill_negative_alts=False,
)
exporter.export(
    [],
    [
        PointOfInterest(
            id=0,
            label="Radar1",
            type="Rx",
            lat=p.lat,
            lon=p.lon,
            alt=p.alt,
        )
        for p in [p_monostatic1, p_monostatic2, p_monostatic3]
    ],
)

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range
from theia.test_data import get_uetliberg_radar


radar = get_uetliberg_radar(power=2000, frequency=1500, rx_bandwidth=2, diameter=2)
calculate_maximum_monostatic_range(radar, 1.0) / 1e3

In [ ]:
from theia.terrain import PlateauHill


In [ ]:
from theia.export_paraview import ParaviewExporter
from theia.terrain import ConstantSphereTerrainModel


terrain = ConstantSphereTerrainModel(alt=0)

exporter = ParaviewExporter(
    "output/sphere",
    lat_min=34.01624,
    lat_max=55.97380,
    lat_res=0.1,
    lon_min=-11.85964,
    lon_max=38.14561,
    lon_res=0.1,
    terrain_model=terrain,
    elevation_factor=1.0,
)
exporter.export([], [])

In [ ]:
from theia.export_paraview import ParaviewExporter
from theia.terrain import (
    FlatEartWithHillsTerrainModel,
    FlatEarthTerrainModel,
    PlateauHill,
)


# terrain = FlatEarthTerrainModel()
terrain = FlatEartWithHillsTerrainModel(
    plateaus=[
        PlateauHill(
            lat_min=40.0,
            lat_max=45.0,
            lon_min=0.0,
            lon_max=5.0,
            height=10_000,
        )
    ]
)

exporter = ParaviewExporter(
    "./output/flat",
    lat_min=34.01624,
    lat_max=55.97380,
    lat_res=0.01,
    lon_min=-11.85964,
    lon_max=38.14561,
    lon_res=0.01,
    terrain_model=terrain,
    elevation_factor=1.0,
    fill_negative_alts=False,
)
exporter.export([], [])

In [ ]:
from theia.coordinates import CoordinateTransformations


CoordinateTransformations.cartesian_to_geodetic(
    3500900.3133147047, -735180.1903229321, 5262810.433892342
)

In [ ]:
import numpy as np

from theia.distance import R_EARTH


R_EARTH - np.linalg.norm(
    np.array(
        [3500900.3133147047, -735180.1903229321, 5262810.433892342], dtype=np.float128
    )
)

In [ ]:
(
    np.square(3500900.3133147047) / 1e13,
    np.square(735180.1903229321) / 1e13,
    np.square(5262810.433892342) / 1e13,
)

In [ ]:
from functools import cached_property

import numpy as np

from theia.coordinates import CoordinateTransformations
from theia.distance import R_EARTH
from theia.terrain import AbstractTerrainModel
from theia.types import Point


a


In [ ]:
from theia.plotting import plot_profile
from theia.terrain import ConstantSphereTerrainModel, SrtmTerrainModel
from theia.types import Point

terrain = SrtmTerrainModel()
terrain = ConstantSphereTerrainModel(alt=100)

plot_profile(
    Point(lat=47.29413, lon=8.28446, alt=terrain.elevationAt(47.29413, 8.28446)),
    Point(lat=46.01222, lon=7.15267, alt=terrain.elevationAt(46.01222, 7.15267)),
    terrain_model=terrain,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from theia.ellipsoid import Ellipsoid, EllipsoidIntersection

e1 = Ellipsoid(
    p1=(4282300.676063238, 639387.6735549602, 4668806.332152337),
    p2=(4278369.997370845, 644603.2701877941, 4671416.235940153),
    r=27809.251860388176,
)
e2 = Ellipsoid(
    p1=(4282300.676063238, 639387.6735549602, 4668806.332152337),
    p2=(4281690.458270983, 629501.0957207535, 4670408.211252162),
    r=18720.50853567265,
)
ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)


rng = np.random.default_rng(seed=830752)
pe1 = np.vstack([e1.sample_surface_uniformly(rng) for _ in range(500)])
pe2 = np.vstack([e2.sample_surface_uniformly(rng) for _ in range(500)])
pi = np.vstack([ei.sample(rng, n_batch=50) for _ in range(1000)])

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pe1[:, 0], y=pe1[:, 1], z=pe1[:, 2], mode="markers"))
fig.add_trace(go.Scatter3d(x=pe2[:, 0], y=pe2[:, 1], z=pe2[:, 2], mode="markers"))
fig.add_trace(go.Scatter3d(x=pi[:, 0], y=pi[:, 1], z=pi[:, 2], mode="markers"))
fig.update_traces(marker_size=1)

In [ ]:
import time


ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)

N = 100

times = []
for n_batch in [50, 100, 200, 250, 300, 400, 500]:
    ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)
    start = time.perf_counter()
    s = [ei.sample(rng, n_batch=n_batch) for _ in range(N)]
    stop = time.perf_counter()
    times.append(
        {
            "n_batch": n_batch,
            "time": (stop - start) / N,
            "cold_start": True,
        }
    )

    start = time.perf_counter()
    s = [ei.sample(rng, n_batch=n_batch) for _ in range(N)]
    stop = time.perf_counter()
    times.append(
        {
            "n_batch": n_batch,
            "time": (stop - start) / N,
            "cold_start": False,
        }
    )

In [ ]:
import pandas as pd
import seaborn as sns


df = pd.DataFrame(times)
sns.barplot(data=df, x="n_batch", y="time", orient="v", hue="cold_start")

In [ ]:
%%timeit
s = ei.sample(rng, n_batch=100)

In [ ]:
# Idea:
# Place PET receivers so that they have as much line of sight to the
# target trajectory as possible. That means:
# - Sample target waypoints.
# - For each waypoint: Calculate visible points on lat-lon-terrain-grid (binary mask)
# - Sum up the masks for each waypoint.
# - Place sensors at maxima.

# TODO:
# Replace sidc code by "TargetCategory", which has an "archetype" and a SIDC code.
# Open for extension in the future.

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST


lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]

In [ ]:
from theia.terrain import elevationAt


elevationAt(lat, lon)

In [ ]:
import abc
import math

from theia.config import ELEVATION_DATA_DIR
from theia.terrain import interpolate_elevation_tile, load_hgt_file
from theia.types import Point


model = SrtmTerrainModel()

In [ ]:
%%timeit
alt = elevationAt(lat, lon)

In [ ]:
%%timeit
alt = model.elevationAt(lat, lon)

In [ ]:
import numba
import numpy as np


@numba.njit(cache=True, inline="always")
def _norm3(v):
    return np.sqrt(v[0] * v[0] + v[1] * v[1] + v[2] * v[2])


@numba.njit(cache=True, inline="always")
def _orthonormal_pair(e):
    """Two unit vectors perpendicular to unit vector e."""
    if abs(e[0]) < 0.9:
        t = np.array([1.0, 0.0, 0.0])
    else:
        t = np.array([0.0, 1.0, 0.0])
    # e2 = t − (t·e)e  normalised
    dot = t[0] * e[0] + t[1] * e[1] + t[2] * e[2]
    e2 = np.array([t[0] - dot * e[0], t[1] - dot * e[1], t[2] - dot * e[2]])
    e2 /= _norm3(e2)
    # e3 = e × e2
    e3 = np.array(
        [
            e[1] * e2[2] - e[2] * e2[1],
            e[2] * e2[0] - e[0] * e2[2],
            e[0] * e2[1] - e[1] * e2[0],
        ]
    )
    return e2, e3


@numba.njit(cache=True)
def _sample_on_ellipsoid(T, R, brange):
    """
    Sample one point uniformly on the prolate-spheroid surface
    defined by foci T, R and bistatic range brange.
    """
    a = 0.5 * brange
    fv = R - T
    c = 0.5 * _norm3(fv)  # focal half-distance

    if c >= a:  # degenerate (invalid range)
        return 0.5 * (T + R)

    b2 = a * a - c * c
    b = np.sqrt(b2)
    a2 = a * a

    e = fv / (2.0 * c)  # unit major-axis vector
    e2, e3 = _orthonormal_pair(e)
    ctr = 0.5 * (T + R)

    # Rejection sampling for surface-uniform (θ, φ)
    while True:
        cos_t = 2.0 * np.random.random() - 1.0  # uniform-on-sphere proposal
        sin_t = np.sqrt(max(0.0, 1.0 - cos_t * cos_t))
        phi = 2.0 * np.pi * np.random.random()

        # Accept prob = sqrt(b²cos²θ + a²sin²θ) / a
        acc = np.sqrt(b2 * cos_t * cos_t + a2 * sin_t * sin_t) / a
        if np.random.random() <= acc:
            cp = np.cos(phi)
            sp = np.sin(phi)
            x = ctr + (a * cos_t) * e + (b * sin_t * cp) * e2 + (b * sin_t * sp) * e3
            return x

In [ ]:
%%timeit
_sample_on_ellipsoid(np.array((0, 0, 0)), np.array((4000, 0, 0)), 150_000)

In [ ]:
from theia.ellipsoid import Ellipsoid


ellipsoid = Ellipsoid((0, 0, 0), (4000, 0, 0), 150_000)

In [ ]:
%%timeit
points = ellipsoid.sample_surface(1, 1)

In [ ]:
# import datetime

# from theia.detection.pet import suggest_pet_receiver_locations
# from theia.grids import LatLonTerrainGrid
# from theia.test_data import build_single_target_from_Bodensee

# trajectory = build_single_target_from_Bodensee(target_id=0)
# targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))
# target_positions = [t.point for t in targets]

# grid = LatLonTerrainGrid(
#     lat_start=46.78125,
#     lat_stop=48.12577,
#     lon_start=7.17484,
#     lon_stop=9.49275,
#     lat_res=0.01,
#     lon_res=0.01,
# )

# best_points = suggest_pet_receiver_locations(grid, target_positions)

In [ ]:
from theia.config import SIDC_RED_FIXED_WING
from theia.simulation.controllers.waypoint_target_controller import (
    WaypointTargetController,
)
from theia.terrain import elevationAt
from theia.test_data import build_fighter_jet_radar, build_single_target_from_Bodensee
from theia.types import Point, Receiver


def build_pet_only_scenario():
    pet_receiver_positions = [
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
    ]

    pet_receivers: list[Receiver] = []
    for i, pos in enumerate(pet_receiver_positions):
        rx = build_fighter_jet_radar(0, 0, 0).receiver.model_copy(deep=True)
        rx.point = pos
        rx.id = i
        pet_receivers.append(rx)

    trajectory = build_single_target_from_Bodensee(target_id=0)
    red_controller = WaypointTargetController.from_trajectory(
        trajectory=trajectory,
        name="Emitting target",
        sidc=SIDC_RED_FIXED_WING,
    )

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
for i, point in enumerate(best_points):
    folium.Marker(
        (point[0], point[1]),
        tooltip=f"Point index = {i}<br />Lat={point[0]:.4f} °<br />Lon={point[1]:.4f} °",
    ).add_to(map)
map

In [ ]:
plt.imshow(n_los)

In [ ]:
%%timeit
los = has_line_of_sight(
    target_pos,
    Point(lat=point[0], lon=point[1], alt=point[2]),
    60,
)

In [ ]:
from theia.coverage import calculate_coverage


calculate_coverage

In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("log.json")

In [ ]:
loader.red_monostatic_radar_detections

In [ ]:
import datetime

from theia.coordinates import POSITIONS_OF_INTEREST
from theia.test_data import (
    build_fighter_jet_radar,
    build_flores_monostatic_radar,
    build_single_target_from_Bodensee,
)
from theia.types import Point


trajectory = build_single_target_from_Bodensee()
target = trajectory(
    datetime.datetime(
        year=2026,
        month=4,
        day=29,
        hour=0,
        minute=0,
        second=58,
    )
)
red_sensor = build_fighter_jet_radar(0, 0, 0)
red_sensor.receiver.point = target.point
red_sensor.transmitter.point = target.point
blue_sensor = build_flores_monostatic_radar(
    Point(
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    1,
    1,
    1,
)

In [ ]:
from theia.line_of_sight import has_line_of_sight


has_line_of_sight(red_sensor.receiver.point, blue_sensor.receiver.point, 30)

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range


r_max_red = calculate_maximum_monostatic_range(red_sensor, 1.0)
r_max_blue = calculate_maximum_monostatic_range(blue_sensor, 1.0)

In [ ]:
from theia.coverage import calculate_coverage


coverage_red = calculate_coverage(
    red_sensor.receiver.point, r_max_red, blue_sensor.receiver.alt, d_theta=2
)
coverage_blue = calculate_coverage(
    blue_sensor.receiver.point, r_max_blue, 1000.0, d_theta=2
)

In [ ]:
from theia.mapping import RadarMap


RadarMap(
    sensors={
        "RED": red_sensor,
        "BLUE": blue_sensor,
    },
    polygons={
        "Coverage RED": coverage_red,
        # "Coverage BLUE": coverage_blue,
    },
).to_map()

In [ ]:
from theia.plotting import plot_profile


plot_profile(
    target.point,
    blue_sensor.receiver.point,
    "RED sensor",
    "BLUE sensor (target)",
)

In [ ]:
radar = loader.red_monostatic_radars[0]

In [ ]:
%%timeit
res = calculate_coverage(radar.receiver.point, r_max, 0.0)